In [1]:
from pathlib import Path
import os

# set project root (one level above notebook for example)
PROJECT_ROOT = Path().resolve().parent

# change working directory
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)


from dotenv import load_dotenv
load_dotenv()

Project root: /root/dev/ledgerx/ledgerx-api


True

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
#from utils.bill_preprocessing import extract_bill_fields
from utils.bill_preprocessing import decrypt_to_temp, get_text_from_pdf
from utils.password_crypto import encrypt_password
#from utils.field_extractor import load_model
#get_text_from_pdf, decrypt_to_temp
from utils.pattern_field_extractor import pattern_field_extraction
from utils.bill_preprocessing import preprocess_statement_text

/root/dev/ledgerx/ledgerx-api/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
value = {
    "name": "BPI",
    "encrypted_password": encrypt_password("20Oct1997814614"),
    "useful_page": [1],
    'bills_path': "./temp_attachments/HSBC Gold - February 2026.pdf"
}

#response = extract_bill_fields("./temp_attachments/BPI Rewards - February 2026.pdf", password="19971020", lang="eng")
dec_path = decrypt_to_temp(value)
#text = get_text_from_pdf(dec_path,lang='eng')
#pre_processed_text = preprocess_statement_text(text)
#print(pre_processed_text)

Pages to extract: [1]
PDF has 4 page(s). Extracting pages: [1]


In [18]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image

doc = fitz.open(str(dec_path))

In [19]:
page = doc[0]

In [24]:
type(page)

pymupdf.Page

In [22]:
import io
import re
import string
from PIL import Image, ImageFilter, ImageOps

def preprocess_image_for_ocr(image: Image.Image) -> Image.Image:
    img = image.convert("L")  # grayscale
    img = ImageOps.autocontrast(img)
    img = img.filter(ImageFilter.SHARPEN)

    # simple threshold
    img = img.point(lambda p: 255 if p > 180 else 0)
    return img

def extract_text_from_page(page: fitz.Page) -> str:
    return page.get_text("text").strip()

def render_page_to_pil(page: fitz.Page, dpi: int = 200) -> Image.Image:
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def ocr_page(image: Image.Image, lang: str = "eng", psm: int = 0) -> str:
    config = f"--psm {psm}"
    text = pytesseract.image_to_string(image, lang=lang, config=config)
    return text.strip()

def is_text_sufficient(text: str, min_chars: int = 50) -> bool:
    if not text:
        return False

    text = text.strip()

    # 1. Length check (basic)
    if len(text) < min_chars:
        return False

    # 2. Remove whitespace
    text_no_space = re.sub(r"\s+", "", text)

    # 3. Ratio of printable ASCII chars
    printable_ratio = sum(c in string.printable for c in text_no_space) / max(len(text_no_space), 1)

    # 4. Ratio of alphabetic chars
    alpha_ratio = sum(c.isalpha() for c in text_no_space) / max(len(text_no_space), 1)

    # 5. Detect "word-like" tokens (at least 3 letters)
    words = re.findall(r"[A-Za-z]{3,}", text)
    word_count = len(words)

    # ---- thresholds (tunable) ----
    if printable_ratio < 0.7:
        return False

    if alpha_ratio < 0.3:
        return False

    if word_count < 5:
        return False

    return True

def normalize_whitespace(text: str) -> str:
    text = text.replace("\x0c", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


ocr_lang: str = "eng"
ocr_psm: int = 4
min_text_chars: int = 50
dpi: int = 200

text = extract_text_from_page(page)

skip = True
if is_text_sufficient(text, min_chars=min_text_chars) and skip:
    print("tanga")
    final_text = normalize_whitespace(text)
else:
    # 2) Fallback to OCR
    image = render_page_to_pil(page, dpi=dpi)
    image = preprocess_image_for_ocr(image)
    ocr_text = ocr_page(image, lang=ocr_lang, psm=ocr_psm)
    final_text = normalize_whitespace(ocr_text)

print(final_text)

tanga
CONTACT US
Customer Service
(02) 8858 0000
From Overseas
63 2 7976 8000
 `
Changes in your personal information? Call us or visit us at www.hsbc.com.ph
HSBC GOLD VISA 4028-9391-1881-4614
Total Due
Minimum Payment
Payment Due Date
Amount Paid
14,753.58
1,500.00
05 Mar 2026
P
For your convenience, please make your payment via internet banking or other channels on the following page.
Amount in words
Cheque No. /
Cheque Date
Bank Name /
Branch Name
The Hongkong and Shanghai Banking Corporation Limited
Card Products Centre, PO BOX 1096 Makati Central Post Office
1250 Makati City, Metro Manila, Philippines
MR JOSHUA CANTOR
BLK 73 LOT 4B DAISY STREET CORNER
SUNFLOWER STREET CAMARIN BRGY 177
CAMARIN N CALOOCAN CITY
METRO MANILA PH 1422
PLEASE DETACH AND RETURN THIS LOWER PORTION TOGETHER WITH YOUR PAYMENT. 
Page 1 of 2
HSBC GOLD VISA
The Hongkong and Shanghai Banking Corporation Limited
Card Products Centre, PO BOX 1096 Makati Central Post Office, 1250 Makati City, Metro Manila, Philippi

In [23]:
pre_processed_text = preprocess_statement_text(final_text)
output = pattern_field_extraction(pre_processed_text)
output

{'total_balance': '14,753.58'}
{'total_balance': '14,753.58', 'due_date': '05 Mar 2026'}
{'total_balance': '14,753.58', 'due_date': '05 Mar 2026', 'min_payment': '1,500.00'}
{'total_balance': '14,753.58', 'due_date': '05 Mar 2026', 'min_payment': '1,500.00', 'credit_limit': '58,000.00'}
{'total_balance': '14,753.58', 'due_date': '05 Mar 2026', 'min_payment': '1,500.00', 'credit_limit': '58,000.00', 'statement_date': '12 FEB 2026'}
{'total_balance': '14,753.58', 'due_date': '05 Mar 2026', 'min_payment': '1,500.00', 'credit_limit': '58,000.00', 'statement_date': '12 FEB 2026', 'customer_number': '4028-9391-1881-4614'}
Date '12 FEB 2026' does not match format '%Y-%m-%d'
Date '12 FEB 2026' does not match format '%B %d, %Y'
Date '12 FEB 2026' does not match format '%b %d, %Y'
Date '12 FEB 2026' does not match format '%m/%d/%Y'
Date '12 FEB 2026' does not match format '%d %B %Y'
Date '05 Mar 2026' does not match format '%Y-%m-%d'
Date '05 Mar 2026' does not match format '%B %d, %Y'
Date '05 

{'customer_number': '4028-XXXX-XXXX-4614',
 'statement_date': '2026-02-12',
 'credit_limit': Decimal('58000.00'),
 'total_amount_due': Decimal('14753.58'),
 'minimum_amount_due': Decimal('1500.00'),
 'payment_due_date': '2026-03-05',
 'source_layout': 'string_strict_sequence'}

In [14]:
pre_processed_text

'BPI Credit Cards Statement of Account\n\nPrepared for\nT MMTTW (QM ENGUOn an CUSTOMER NUMBER 020100-4-10-7956071\nUNA A\n\nJOSHUA Z CANTOR STATEMENT DATE JANUARY 28, 2026\n\nBlk 2 Lot 14 Peter Street\nCielito Homes Brgy 177 CREDIT LIMIT 314,000.00\n\n1422 Caloocan City TOTAL AMOUNT DUE 20,958.15\nMINIMUM AMOUNT DUE 850.00\n\nPAYMENT DUE DATE FEBRUARY 18, 2026\n\nPrevious (-) Payments / | (+) Purchases | (+) Installment | (+) Finance (+) Late Payment\nCard Type Balance Credits and and Advances | Due Charges and | Charges Amount Due\nRebates Other Fees\n\nBPI Rewards 20,946.05 20,946.05 1,353.96 19,604.19 20,958.15\n\n20,946.05 20,946.05 1,353.96 19,604.19 20,958.15\n\nPast Due Amount 0.00\nMinimum Amount Due 850.00\nUnhbilled Installment Amount $29,282.81\nTotal Outstanding Balance $50,240.96\n\nREWARDS\n\nTo view your BPI Points balance earned from your BPI Credit Card/s and other BPI Product/s you may have, simply download the VYBE app and register now!\n\nPlease use your customer nu